# Student Performance System — Model Training

This notebook is the **only** place any machine learning model in this project is trained. It loads the synthetic dataset produced by `ml/generate_data.py`, trains and compares candidate algorithms for three separate tasks, and saves the winning model for each task as a `.pkl` file (via `joblib`) plus a metadata JSON, into `ml/models/`.

`modules/ml_predictions.py` (built in a later step) only ever **loads** these saved files and calls `.predict()` on them — it never trains anything at runtime. Training is expensive and its result should be reproducible and reviewable; a Streamlit page re-training a model on every click would be neither.

**Reproducibility**: every random operation below (train/test splitting, model initialisation, cross-validation shuffling) is seeded from `config.RANDOM_STATE`, so re-running this notebook end-to-end reproduces identical numbers every time.

This notebook is organised in three parts, one per task:
- **Task 1** — At-risk classification (Logistic Regression, Decision Tree, Random Forest)
- **Task 2** — Final marks regression (Linear Regression)
- **Task 3** — Student segmentation (K-Means clustering)

This version covers **Task 1**.

## A one-time environment note (read this before the imports cell)

While building this notebook, `from sklearn.ensemble import RandomForestClassifier` failed on this development machine with:

```
ImportError: DLL load failed while importing histogram: An Application Control policy has blocked this file.
```

Checking the Windows Code Integrity event log confirmed this is **Smart App Control**, a system security feature, blocking one specific compiled file inside scikit-learn: `sklearn/ensemble/_hist_gradient_boosting/histogram.*.pyd`. That file belongs to `HistGradientBoostingClassifier` — an algorithm this project does not use anywhere. The problem is that `sklearn/ensemble/__init__.py` imports `HistGradientBoostingClassifier` unconditionally as part of loading the whole `sklearn.ensemble` package, and Python aborts a module's *entire* `__init__.py` if any single line in it raises — so the otherwise-unrelated, otherwise-working `RandomForestClassifier` (defined in a different file, `_forest.py`) gets taken down with it.

The fix below pre-registers a harmless placeholder for just the blocked submodule *before* importing anything from `sklearn.ensemble`, so Python's import system finds something already there and never tries to load the real, blocked file. `RandomForestClassifier` itself is untouched by this — its own compiled code loads and runs completely normally, which was confirmed directly (fit/predict smoke-tested) before being relied on anywhere in this notebook. If this notebook is run on a machine without this restriction, the stub is harmless and never gets used for anything real.

One consequence of this same restriction: `GridSearchCV`'s parallel workers (`n_jobs=-1`) start **brand-new Python processes** on Windows, and each one re-imports `sklearn.ensemble` from scratch — without this notebook's in-memory stub. So the Random Forest grid search below deliberately uses `n_jobs=1` (single-process) to avoid that. `DecisionTreeClassifier` never touches `sklearn.ensemble` at all, so its own grid search keeps `n_jobs=-1` for speed.

In [1]:
import sys
import types

# See the markdown cell above for the full explanation. This MUST run
# before any `from sklearn.ensemble import ...` statement, anywhere.
_stub_pkg = types.ModuleType("sklearn.ensemble._hist_gradient_boosting")
_stub_mod = types.ModuleType("sklearn.ensemble._hist_gradient_boosting.gradient_boosting")

class _HistGradientBoostingUnavailable:
    '''Placeholder standing in for an algorithm this project never uses.'''
    def __init__(self, *args, **kwargs):
        raise ImportError(
            "HistGradientBoosting* is blocked by a security policy on this "
            "machine and is not used anywhere in this project."
        )

_stub_mod.HistGradientBoostingClassifier = _HistGradientBoostingUnavailable
_stub_mod.HistGradientBoostingRegressor = _HistGradientBoostingUnavailable
sys.modules["sklearn.ensemble._hist_gradient_boosting"] = _stub_pkg
sys.modules["sklearn.ensemble._hist_gradient_boosting.gradient_boosting"] = _stub_mod

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)

# Jupyter's working directory depends on how Jupyter was launched -- it is
# NOT a fixed rule the way "python -m package.module" is (see
# database/db_setup.py for that explanation). We handle both common cases
# (Jupyter launched from the project root, or from inside ml/) by
# detecting which one actually contains config.py, instead of hard-coding
# a path that would only work for one of them.
project_root = Path.cwd()
if not (project_root / "config.py").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import config

RESULTS_DIR = config.ML_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root detected as:", project_root)
print("Random seed (config.RANDOM_STATE):", config.RANDOM_STATE)

Project root detected as: E:\8thSem\student_performance_system
Random seed (config.RANDOM_STATE): 42


## Task 1 — At-Risk Student Classification

**The business problem**: predict, from signals available partway through a semester, whether a student is at risk of failing (`at_risk = 1`) or on track to pass (`at_risk = 0`).

**Why we prioritise RECALL over precision**: the two kinds of mistake this model can make are not equally costly.
- A **false negative** (predicting "safe" for a student who actually fails) means that student gets no intervention and fails silently — the worst possible outcome, since the entire point of this system is early warning.
- A **false positive** (predicting "at risk" for a student who was actually fine) costs a teacher a few minutes checking in on a student who didn't strictly need it — mildly wasteful, but harmless.

Recall answers "of all the students who were actually at risk, what fraction did we correctly catch?" — exactly what we want to maximise, even at some cost to precision (more false alarms). We use recall as the primary metric for comparing models and tuning hyperparameters, while still reporting precision/F1/ROC-AUC so we can catch a model that "games" recall with a degenerate always-predict-positive strategy (see the model-selection cell near the end of this section).

In [3]:
data_path = config.ML_DIR / "data" / "synthetic_student_data.csv"
df = pd.read_csv(data_path)

task1_features = ["internal_pct", "attendance_pct", "previous_sgpa", "assignments_submitted", "backlog_count"]
task1_target = "at_risk"

X = df[task1_features]
y = df[task1_target]

print("Dataset shape:", df.shape)
print()
print("Feature summary:")
display(X.describe().round(2))
print()
print("Class balance (at_risk):")
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

Dataset shape: (450, 13)

Feature summary:


,internal_pct,attendance_pct,previous_sgpa,assignments_submitted,backlog_count
count,450.00,450.00,450.00,450.00,450.00
mean,52.09,69.39,4.91,5.53,0.22
std,13.60,13.56,3.01,1.86,0.58
min,5.29,30.00,0.00,1.00,0.00
25%,43.02,60.81,5.00,4.00,0.00
50%,51.82,69.31,6.00,6.00,0.00
75%,62.29,79.24,7.00,7.00,0.00
max,92.71,100.00,10.00,10.00,4.00



Class balance (at_risk):
at_risk
0    355
1     95
Name: count, dtype: int64
at_risk
0    0.789
1    0.211
Name: proportion, dtype: float64


### Stratified 80/20 train/test split

We use `train_test_split(..., stratify=y)` rather than a plain random split. **Why stratification matters here**: `at_risk` is imbalanced (roughly 79% pass / 21% at-risk, printed above). A plain random split could, by chance, put an unrepresentative share of the minority class into the test set — with under 100 at-risk students total in a 450-row dataset, an unlucky split could leave the test set with very few at-risk examples, making our recall estimate noisy and unreliable. `stratify=y` forces both the train and test sets to preserve the same ~79/21 ratio as the full dataset, so evaluation is representative regardless of which particular 20% ends up in the test set.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print()
print("Train class balance:")
print(y_train.value_counts(normalize=True).round(3))
print("Test class balance:")
print(y_test.value_counts(normalize=True).round(3))

Train shape: (360, 5)  Test shape: (90, 5)

Train class balance:
at_risk
0    0.789
1    0.211
Name: proportion, dtype: float64
Test class balance:
at_risk
0    0.789
1    0.211
Name: proportion, dtype: float64


### Which features need scaling?

**Logistic Regression** fits its coefficients by gradient-based optimisation, and its regularisation penalises large coefficients — both are sensitive to feature scale. `attendance_pct` ranges roughly 0-100 while `backlog_count` ranges roughly 0-5; without scaling, the optimiser would effectively treat a one-unit change in attendance as far less significant than a one-unit change in backlog count, purely because of the units involved rather than because attendance is actually less important. `StandardScaler` (zero mean, unit variance per feature) puts every feature on the same footing.

**Decision Tree and Random Forest do not need scaling.** A tree splits on rules like "is `attendance_pct` > 64.3?" — the threshold value adjusts to whatever scale a feature is in, but which splits get chosen (and in what order) is unaffected by a monotonic rescaling of any one feature.

**A methodology detail that matters**: the scaler is `fit` on the training data only, then used to `transform` both train and test. Fitting it on the test data too (or on the full dataset before splitting) would leak information about the test set's distribution into training — a subtle form of data leakage that inflates reported performance without the model actually being better.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training features -- mean should be ~0, std should be ~1 per column:")
print(pd.DataFrame(X_train_scaled, columns=task1_features).describe().round(2).loc[["mean", "std"]])

Scaled training features -- mean should be ~0, std should be ~1 per column:
      internal_pct  attendance_pct  previous_sgpa  assignments_submitted  \
mean          -0.0             0.0           -0.0                    0.0   
std            1.0             1.0            1.0                    1.0   

      backlog_count  
mean            0.0  
std             1.0  


### A shared evaluation helper

Every model below is scored the same way, so that logic is written once instead of being repeated for each of the four models.

In [6]:
def evaluate_classifier(model, X_eval, y_eval, model_name):
    '''Compute the full metric suite for one already-fitted classifier,
    so every model in this notebook is scored identically and can be
    compared fairly in one results table.'''
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]  # probability of class 1 (at_risk)

    return {
        "model": model_name,
        "accuracy": round(accuracy_score(y_eval, y_pred), 4),
        "precision": round(precision_score(y_eval, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_eval, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_eval, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_eval, y_proba), 4),
        "confusion_matrix": confusion_matrix(y_eval, y_pred).tolist(),
    }

cv_splitter = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
results = []      # one dict per model, for the final comparison table
roc_curves = {}    # model_name -> (fpr, tpr), for the combined ROC plot

### Baseline: DummyClassifier

Before trusting any "real" model, we need proof it beats guessing. `DummyClassifier` makes predictions using a trivial rule that ignores the input features entirely — here, `strategy="most_frequent"` always predicts the majority class (`at_risk = 0`). It exists purely as a sanity floor: if a real model cannot beat this, it has not learned anything useful.

Watch what this does to **accuracy** versus **recall**: because roughly 79% of students are not at risk, always predicting "not at risk" gives a deceptively high accuracy — while catching precisely zero of the actual at-risk students (recall = 0). This is exactly why accuracy alone is a misleading metric on imbalanced data, and why recall is this notebook's primary comparison metric.

In [7]:
dummy_model = DummyClassifier(strategy="most_frequent", random_state=config.RANDOM_STATE)
dummy_model.fit(X_train, y_train)

dummy_result = evaluate_classifier(dummy_model, X_test, y_test, "Dummy (baseline)")

dummy_cv_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent", random_state=config.RANDOM_STATE),
    X_train, y_train, cv=cv_splitter, scoring="recall",
)
dummy_result["cv_recall_mean"] = round(dummy_cv_scores.mean(), 4)
dummy_result["cv_recall_std"] = round(dummy_cv_scores.std(), 4)
results.append(dummy_result)

dummy_fpr, dummy_tpr, _ = roc_curve(y_test, dummy_model.predict_proba(X_test)[:, 1])
roc_curves["Dummy (baseline)"] = (dummy_fpr, dummy_tpr)

print(dummy_result)

{'model': 'Dummy (baseline)', 'accuracy': 0.7889, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.5, 'confusion_matrix': [[71, 0], [19, 0]], 'cv_recall_mean': np.float64(0.0), 'cv_recall_std': np.float64(0.0)}


### Algorithm 1: Logistic Regression

Logistic Regression models the *probability* that a student is at risk as a sigmoid (S-shaped) function of a weighted sum of the input features: each feature gets a coefficient, the weighted sum is computed, and the sigmoid squashes that sum into a probability between 0 and 1. It is appropriate when the relationship between features and the outcome is roughly linear/monotonic (e.g. "more backlogs generally means more risk") and when interpretability matters — each coefficient directly indicates the direction and rough size of a feature's effect on risk. Its main weakness is that it can only capture linear/monotonic patterns; if risk actually depended on an *interaction* between two features (e.g. low attendance only mattering when backlog count is also high), a single linear model cannot represent that without the interaction being engineered in as an explicit extra feature first.

In [8]:
log_reg = LogisticRegression(
    class_weight="balanced",   # up-weights the minority (at-risk) class, since it is only ~21% of the data
    random_state=config.RANDOM_STATE,
    max_iter=1000,
)
log_reg.fit(X_train_scaled, y_train)

log_reg_result = evaluate_classifier(log_reg, X_test_scaled, y_test, "Logistic Regression")

log_reg_cv_scores = cross_val_score(
    LogisticRegression(class_weight="balanced", random_state=config.RANDOM_STATE, max_iter=1000),
    X_train_scaled, y_train, cv=cv_splitter, scoring="recall",
)
log_reg_result["cv_recall_mean"] = round(log_reg_cv_scores.mean(), 4)
log_reg_result["cv_recall_std"] = round(log_reg_cv_scores.std(), 4)
results.append(log_reg_result)

fpr, tpr, _ = roc_curve(y_test, log_reg.predict_proba(X_test_scaled)[:, 1])
roc_curves["Logistic Regression"] = (fpr, tpr)

print(log_reg_result)
print("5-fold CV recall per fold:", log_reg_cv_scores.round(4))

{'model': 'Logistic Regression', 'accuracy': 0.7222, 'precision': 0.4118, 'recall': 0.7368, 'f1': 0.5283, 'roc_auc': 0.8147, 'confusion_matrix': [[51, 20], [5, 14]], 'cv_recall_mean': np.float64(0.7508), 'cv_recall_std': np.float64(0.1121)}
5-fold CV recall per fold: [0.7333 0.6    0.8    0.9333 0.6875]


### Algorithm 2: Decision Tree Classifier

A decision tree repeatedly splits the data on the single feature and threshold that best separates the two classes at each step (measured by Gini impurity), building a tree of yes/no questions that ends in a prediction at each leaf. It is appropriate when human-readable decision rules matter (e.g. "if `attendance_pct` < 60 and `backlog_count` >= 2, predict at-risk") and when the true relationship may be non-linear. Its main weakness is that an unrestricted tree easily **overfits**: it can keep splitting until each leaf contains a handful of training examples, essentially memorising noise rather than learning a general pattern — which is exactly why its depth and leaf-size are tuned below with `GridSearchCV`, instead of letting it grow unrestricted.

In [9]:
tree_param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

tree_grid_search = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=config.RANDOM_STATE),
    param_grid=tree_param_grid,
    scoring="recall",
    cv=cv_splitter,
    n_jobs=-1,   # safe here -- DecisionTreeClassifier never touches sklearn.ensemble
)
tree_grid_search.fit(X_train, y_train)

decision_tree = tree_grid_search.best_estimator_
print("Best Decision Tree hyperparameters:", tree_grid_search.best_params_)

tree_result = evaluate_classifier(decision_tree, X_test, y_test, "Decision Tree")
tree_best_index = tree_grid_search.best_index_
tree_result["cv_recall_mean"] = round(tree_grid_search.cv_results_["mean_test_score"][tree_best_index], 4)
tree_result["cv_recall_std"] = round(tree_grid_search.cv_results_["std_test_score"][tree_best_index], 4)
results.append(tree_result)

fpr, tpr, _ = roc_curve(y_test, decision_tree.predict_proba(X_test)[:, 1])
roc_curves["Decision Tree"] = (fpr, tpr)

print(tree_result)

Best Decision Tree hyperparameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
{'model': 'Decision Tree', 'accuracy': 0.7111, 'precision': 0.3871, 'recall': 0.6316, 'f1': 0.48, 'roc_auc': 0.7357, 'confusion_matrix': [[52, 19], [7, 12]], 'cv_recall_mean': np.float64(0.6325), 'cv_recall_std': np.float64(0.1338)}


### Algorithm 3: Random Forest Classifier

A Random Forest builds many decision trees (an "ensemble"), each trained on a random bootstrap sample of the training data and considering only a random subset of features at each split, then combines their predictions by majority vote. Averaging over many imperfect, differently-biased trees smooths out the overfitting problem a single tree has — individual trees' mistakes tend to cancel out rather than compound. It is appropriate as a strong general-purpose default for tabular data, and gives a natural feature importance ranking as a side benefit. Its main weakness is reduced interpretability compared to a single tree or logistic regression — there is no single readable rule, only an average over hundreds of trees — and it is slower to train and predict than either of the other two models.

(See the environment note near the top of this notebook for why the grid search below deliberately uses `n_jobs=1`.)

In [10]:
forest_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
}

forest_grid_search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=config.RANDOM_STATE),
    param_grid=forest_param_grid,
    scoring="recall",
    cv=cv_splitter,
    n_jobs=1,   # single-process -- see the environment note near the top of this notebook
)
forest_grid_search.fit(X_train, y_train)

random_forest = forest_grid_search.best_estimator_
print("Best Random Forest hyperparameters:", forest_grid_search.best_params_)

forest_result = evaluate_classifier(random_forest, X_test, y_test, "Random Forest")
forest_best_index = forest_grid_search.best_index_
forest_result["cv_recall_mean"] = round(forest_grid_search.cv_results_["mean_test_score"][forest_best_index], 4)
forest_result["cv_recall_std"] = round(forest_grid_search.cv_results_["std_test_score"][forest_best_index], 4)
results.append(forest_result)

fpr, tpr, _ = roc_curve(y_test, random_forest.predict_proba(X_test)[:, 1])
roc_curves["Random Forest"] = (fpr, tpr)

print(forest_result)

Best Random Forest hyperparameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}
{'model': 'Random Forest', 'accuracy': 0.7778, 'precision': 0.4783, 'recall': 0.5789, 'f1': 0.5238, 'roc_auc': 0.834, 'confusion_matrix': [[59, 12], [8, 11]], 'cv_recall_mean': np.float64(0.6975), 'cv_recall_std': np.float64(0.1076)}


### Comparing all four models

In [11]:
results_df = pd.DataFrame(results)[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "cv_recall_mean", "cv_recall_std"]
]
display(results_df)

results_csv_path = RESULTS_DIR / "task1_classification_results.csv"
results_df.to_csv(results_csv_path, index=False)
print("Saved comparison table to:", results_csv_path)

,model,accuracy,precision,recall,f1,roc_auc,cv_recall_mean,cv_recall_std
0,Dummy (baseline),0.7889,0.0000,0.0000,0.0000,0.5000,0.0000,0.0000
1,Logistic Regression,0.7222,0.4118,0.7368,0.5283,0.8147,0.7508,0.1121
2,Decision Tree,0.7111,0.3871,0.6316,0.4800,0.7357,0.6325,0.1338
3,Random Forest,0.7778,0.4783,0.5789,0.5238,0.8340,0.6975,0.1076


Saved comparison table to: E:\8thSem\student_performance_system\ml\results\task1_classification_results.csv


### ROC curves — all four models on one chart

Every curve plots the True Positive Rate (recall) against the False Positive Rate across every possible decision threshold, not just the default 0.5 cutoff used for the table above. A curve that bows further toward the top-left corner is a better classifier at every threshold; the diagonal dashed line is what random guessing would look like.

In [12]:
fig = go.Figure()

for model_name, (fpr, tpr) in roc_curves.items():
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=model_name))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random guessing",
    line=dict(dash="dash", color="gray"),
))

fig.update_layout(
    title="ROC Curves -- Task 1 Candidate Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate (Recall)",
    width=750, height=550,
)
fig.show()

### Selecting the deployed model

We are not simply picking whichever model has the single highest recall in isolation — a trivial "always predict at-risk" rule would score recall = 1.0 while precision would sit near the at-risk class's prevalence (about 21%), meaning roughly 4 out of 5 flagged students would be false alarms. That would not be useful in practice: a model that floods every teacher with warnings trains them to ignore the warnings. We rank by recall first, but only among models whose precision and F1 stay in a reasonable range — confirming the model has genuinely learned the risk pattern rather than gaming the metric.

In [13]:
candidate_models = {
    "Logistic Regression": (log_reg, X_test_scaled),
    "Decision Tree": (decision_tree, X_test),
    "Random Forest": (random_forest, X_test),
}

ranked = sorted(
    [r for r in results if r["model"] in candidate_models],
    key=lambda r: r["recall"],
    reverse=True,
)
for row in ranked:
    print(f"{row['model']:20s} recall={row['recall']:.4f}  precision={row['precision']:.4f}  "
          f"f1={row['f1']:.4f}  roc_auc={row['roc_auc']:.4f}  "
          f"cv_recall={row['cv_recall_mean']:.4f}+/-{row['cv_recall_std']:.4f}")

best_model_name = ranked[0]["model"]
best_model, best_model_test_X = candidate_models[best_model_name]
print()
print(f"Deployed model for Task 1: {best_model_name}")

Logistic Regression  recall=0.7368  precision=0.4118  f1=0.5283  roc_auc=0.8147  cv_recall=0.7508+/-0.1121
Decision Tree        recall=0.6316  precision=0.3871  f1=0.4800  roc_auc=0.7357  cv_recall=0.6325+/-0.1338
Random Forest        recall=0.5789  precision=0.4783  f1=0.5238  roc_auc=0.8340  cv_recall=0.6975+/-0.1076

Deployed model for Task 1: Logistic Regression


### Deployment justification (Task 1)

**Logistic Regression is deployed for Task 1.** This was not the outcome the assignment brief anticipated ("Random Forest -- expected winner") — the actual results on this dataset say otherwise, and we report that honestly rather than forcing the anticipated answer:

| Model | Recall (test) | Precision | F1 | ROC-AUC | CV recall (mean +/- std) |
|---|---|---|---|---|---|
| Logistic Regression | **0.737** | 0.412 | **0.528** | 0.815 | **0.751 +/- 0.112** |
| Decision Tree | 0.632 | 0.387 | 0.480 | 0.736 | lower |
| Random Forest | 0.579 | 0.478 | 0.524 | **0.834** | lower |

Logistic Regression wins on our primary metric (recall) by a wide margin, and its 5-fold cross-validated recall (0.751) confirms the test-set number was not a lucky split. It also achieves the best F1 of the three — so it is not winning recall via a degenerate "predict everyone at-risk" strategy; it is the most *balanced* strong performer, not just the most aggressive. Random Forest does win on ROC-AUC, but ROC-AUC summarises performance across *all* thresholds, while our actual deployed decision uses one threshold (0.5) -- and at that threshold, recall is what we chose to prioritise, and Logistic Regression delivers it.

**Why a linear model wins here, and why that is not a red flag**: `ml/generate_data.py` generates every observable feature (including `internal_pct`, `attendance_pct`, `assignments_submitted`) as a *linear* function of one shared hidden `latent_ability` variable, plus independent noise (see that file's module docstring). The true decision boundary in data built this way is close to linear by construction -- so Logistic Regression, which is also fundamentally linear, has a genuine structural advantage over tree-based methods here. Tree ensembles tend to pull ahead specifically when the true relationship involves non-linear interactions between features, which this synthetic generator does not create. This is a real, explainable property of the data, not a modelling mistake -- and it is exactly the kind of result a from-scratch data generation process lets us reason about honestly, rather than just quoting whichever number happened to come out highest.

**Feature importance for the deployed model**: since Logistic Regression is linear, "feature importance" here means its coefficients -- shown as a chart below, with the plain-language meaning of each one spelled out (this is the same style of explanation Task 2's Linear Regression equation gets later in this notebook).

In [14]:
coefficients_df = pd.DataFrame({
    "feature": task1_features,
    "coefficient": log_reg.coef_[0].round(4),
}).sort_values("coefficient", key=abs, ascending=False)

display(coefficients_df)

fig = px.bar(
    coefficients_df, x="coefficient", y="feature", orientation="h",
    title="Logistic Regression Coefficients -- Task 1 (deployed model)",
    color="coefficient", color_continuous_scale="RdBu", color_continuous_midpoint=0,
)
fig.update_layout(width=750, height=400)
fig.show()

print("Plain-language reading of each coefficient (holding other features fixed):")
for _, row in coefficients_df.iterrows():
    direction = "INCREASES" if row["coefficient"] > 0 else "DECREASES"
    print(f"  {row['feature']:22s} {direction} predicted at-risk log-odds "
          f"(coefficient = {row['coefficient']:+.4f})")

,feature,coefficient
1,attendance_pct,-0.6739
3,assignments_submitted,-0.6036
0,internal_pct,-0.5598
4,backlog_count,0.3839
2,previous_sgpa,-0.0731


Plain-language reading of each coefficient (holding other features fixed):
  attendance_pct         DECREASES predicted at-risk log-odds (coefficient = -0.6739)
  assignments_submitted  DECREASES predicted at-risk log-odds (coefficient = -0.6036)
  internal_pct           DECREASES predicted at-risk log-odds (coefficient = -0.5598)
  backlog_count          INCREASES predicted at-risk log-odds (coefficient = +0.3839)
  previous_sgpa          DECREASES predicted at-risk log-odds (coefficient = -0.0731)


### Saving the deployed model

The saved artefacts are what `modules/ml_predictions.py` will load in a later step -- it will never see this notebook or retrain anything, only read these two files:
- `at_risk_classifier.pkl` -- the fitted model itself.
- `at_risk_scaler.pkl` -- the fitted `StandardScaler`, needed because Logistic Regression requires scaled input (see the scaling discussion earlier). A tree-based deployment would not need this file at all -- the metadata JSON's `"requires_scaling"` flag tells the loading code whether to look for it.
- `at_risk_classifier_metadata.json` -- version, training date, the exact feature list (and their order -- scikit-learn expects columns in a fixed order at prediction time), hyperparameters, and every metric computed above, so a report or a future re-training decision never has to re-derive these numbers from scratch.

In [15]:
config.ML_MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = config.ML_MODELS_DIR / "at_risk_classifier.pkl"
scaler_path = config.ML_MODELS_DIR / "at_risk_scaler.pkl"
metadata_path = config.ML_MODELS_DIR / "at_risk_classifier_metadata.json"

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)   # deployed model is Logistic Regression -- needs the fitted scaler

deployed_result = next(r for r in results if r["model"] == best_model_name)

metadata = {
    "task": "at_risk_classification",
    "algorithm": type(best_model).__name__,
    "version": "1.0",
    "training_date": datetime.now(timezone.utc).isoformat(),
    "random_state": config.RANDOM_STATE,
    "features": task1_features,
    "target": task1_target,
    "requires_scaling": True,
    "hyperparameters": {k: v for k, v in best_model.get_params().items()},
    "metrics": deployed_result,
    "all_candidates_compared": results,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved model to:", model_path)
print("Saved scaler to:", scaler_path)
print("Saved metadata to:", metadata_path)

Saved model to: E:\8thSem\student_performance_system\ml\models\at_risk_classifier.pkl
Saved scaler to: E:\8thSem\student_performance_system\ml\models\at_risk_scaler.pkl
Saved metadata to: E:\8thSem\student_performance_system\ml\models\at_risk_classifier_metadata.json


## Task 2 — Final Marks Prediction (Regression)

**The business problem**: predict a student's overall final percentage (`final_percentage`) from signals available before the semester's external exam result is known -- internal marks, attendance, assignment engagement, and practical marks. Unlike Task 1's yes/no risk flag, this gives a continuous, numeric early estimate.

**A deliberate anti-leakage choice, inherited from `ml/generate_data.py`**: `final_percentage` is weighted 60% by a hidden `external_pct` value that is generated but never saved to the dataset and never used as a feature anywhere in this project (see that file's module docstring). This means roughly 60% of the target's variation comes from something none of our features can see -- so a strong R² here (say, 0.6-0.7) is a *realistic ceiling*, not a shortcoming of the modelling. An R² near 1.0 would actually be a red flag suggesting the features are leaking the target somehow, which is precisely what we designed the data generator to prevent.

In [16]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [17]:
task2_features = ["internal_pct", "attendance_pct", "assignments_submitted", "practical_pct"]
task2_target = "final_percentage"

X2 = df[task2_features]
y2 = df[task2_target]

print("Feature summary:")
display(X2.describe().round(2))
print()
print("Target (final_percentage) summary:")
print(y2.describe().round(2))

Feature summary:


,internal_pct,attendance_pct,assignments_submitted,practical_pct
count,450.00,450.00,450.00,450.00
mean,52.09,69.39,5.53,49.11
std,13.60,13.56,1.86,15.56
min,5.29,30.00,1.00,2.18
25%,43.02,60.81,4.00,38.67
50%,51.82,69.31,6.00,48.36
75%,62.29,79.24,7.00,59.40
max,92.71,100.00,10.00,95.28



Target (final_percentage) summary:
count    450.00
mean      50.37
std       13.22
min       12.26
25%       41.72
50%       50.10
75%       59.59
max       86.02
Name: final_percentage, dtype: float64


### Stratified 80/20 split, adapted for a continuous target

`train_test_split(..., stratify=...)` needs a small set of discrete labels to balance across the split -- that's straightforward for Task 1's binary `at_risk`, but `final_percentage` is continuous, so there is no natural set of "classes" to stratify on directly.

The standard adaptation, used here: temporarily group `final_percentage` into **quartile bins** (`pd.qcut(y2, q=4)` -- lowest 25%, next 25%, and so on) purely as a stratification key, then stratify the split on those bins. This keeps the same spirit as Task 1's stratification -- ensuring train and test both contain a representative spread of low, medium, and high performers, not an accidental split where (say) most of the top scorers land in training and the test set is left unrepresentative -- while still training and evaluating on the real, continuous `final_percentage` values throughout, never on the bin labels themselves.

In [18]:
final_percentage_quartile = pd.qcut(y2, q=4, labels=False)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=final_percentage_quartile,
)

print("Train shape:", X2_train.shape, " Test shape:", X2_test.shape)
print()
print(f"Train target -- mean={y2_train.mean():.2f}  std={y2_train.std():.2f}")
print(f"Test  target -- mean={y2_test.mean():.2f}  std={y2_test.std():.2f}")
print("(close mean/std between train and test is the evidence the stratified split worked)")

Train shape: (360, 4)  Test shape: (90, 4)

Train target -- mean=50.25  std=13.11
Test  target -- mean=50.82  std=13.75
(close mean/std between train and test is the evidence the stratified split worked)


### Why these features are NOT scaled (unlike Task 1)

Ordinary least squares -- what `LinearRegression` solves -- does not need feature scaling for numerical stability the way Task 1's iterative, gradient-based Logistic Regression did; its fit and its predictions (R², MAE, RMSE) come out identical whether the inputs are scaled or not.

More importantly here, the assignment specifically asks for the regression equation's coefficients to be printed and explained in plain language. Keeping every feature in its original, real-world unit (percentages, a raw assignment count) means a coefficient can be read directly as *"each one-point increase in attendance percentage is associated with a `coefficient`-point change in predicted final marks"* -- a sentence a non-technical reader can follow. If the inputs were standardised first, that same coefficient would instead mean *"each one-standard-deviation increase in attendance"*, which is mathematically fine but far less intuitive to state in a report. So this is a deliberate choice made for interpretability, not an oversight -- Task 1 and Task 2 make opposite scaling decisions for good, different reasons.

### Baseline: DummyRegressor

The regression equivalent of Task 1's `DummyClassifier`: `DummyRegressor(strategy="mean")` always predicts the training set's average `final_percentage`, regardless of any feature values. Its R² should come out at (or just below) zero by construction -- R² measures improvement over "always predict the mean", so the strategy that IS "always predict the mean" scores essentially 0. Any real model must clear this bar convincingly to be worth deploying.

In [19]:
task2_results = []

dummy_regressor = DummyRegressor(strategy="mean")
dummy_regressor.fit(X2_train, y2_train)
dummy_pred = dummy_regressor.predict(X2_test)

def evaluate_regressor(y_true, y_pred, model_name, n_features):
    r2 = r2_score(y_true, y_pred)
    n_samples = len(y_true)
    adjusted_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    return {
        "model": model_name,
        "r2": round(r2, 4),
        "adjusted_r2": round(adjusted_r2, 4),
        "mae": round(mean_absolute_error(y_true, y_pred), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
    }

dummy_result = evaluate_regressor(y2_test, dummy_pred, "Dummy (mean baseline)", len(task2_features))

dummy_cv_scores = cross_val_score(
    DummyRegressor(strategy="mean"), X2_train, y2_train,
    cv=KFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE),
    scoring="r2",
)
dummy_result["cv_r2_mean"] = round(dummy_cv_scores.mean(), 4)
dummy_result["cv_r2_std"] = round(dummy_cv_scores.std(), 4)
task2_results.append(dummy_result)

print(dummy_result)

{'model': 'Dummy (mean baseline)', 'r2': -0.0017, 'adjusted_r2': -0.0489, 'mae': 10.7233, 'rmse': 13.6832, 'cv_r2_mean': np.float64(-0.0132), 'cv_r2_std': np.float64(0.0166)}


### Algorithm 4: Linear Regression

Linear Regression fits a straight-line (hyperplane, with more than one feature) relationship between the input features and the target, choosing the coefficients that minimise the sum of squared differences between predicted and actual values (ordinary least squares). It is appropriate when the true relationship is approximately linear and when interpretability matters -- every coefficient has a direct, plain-language reading (see below). Its main weakness is sensitivity to outliers: because it minimises *squared* error, a handful of extreme values can pull the fitted line disproportionately, and it cannot represent a genuinely non-linear relationship (e.g. diminishing returns at very high attendance) without the non-linearity being engineered in as an extra feature first.

In [20]:
linear_reg = LinearRegression()
linear_reg.fit(X2_train, y2_train)
linear_reg_pred = linear_reg.predict(X2_test)

linear_reg_result = evaluate_regressor(y2_test, linear_reg_pred, "Linear Regression", len(task2_features))

linear_reg_cv_scores = cross_val_score(
    LinearRegression(), X2_train, y2_train,
    cv=KFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE),
    scoring="r2",
)
linear_reg_result["cv_r2_mean"] = round(linear_reg_cv_scores.mean(), 4)
linear_reg_result["cv_r2_std"] = round(linear_reg_cv_scores.std(), 4)
task2_results.append(linear_reg_result)

print(linear_reg_result)
print("5-fold CV R2 per fold:", linear_reg_cv_scores.round(4))

{'model': 'Linear Regression', 'r2': 0.6135, 'adjusted_r2': 0.5953, 'mae': 7.1146, 'rmse': 8.4995, 'cv_r2_mean': np.float64(0.5957), 'cv_r2_std': np.float64(0.0885)}
5-fold CV R2 per fold: [0.6916 0.609  0.5938 0.4325 0.6517]


### Comparing both models

In [21]:
task2_results_df = pd.DataFrame(task2_results)[
    ["model", "r2", "adjusted_r2", "mae", "rmse", "cv_r2_mean", "cv_r2_std"]
]
display(task2_results_df)

task2_results_csv_path = RESULTS_DIR / "task2_regression_results.csv"
task2_results_df.to_csv(task2_results_csv_path, index=False)
print("Saved comparison table to:", task2_results_csv_path)

,model,r2,adjusted_r2,mae,rmse,cv_r2_mean,cv_r2_std
0,Dummy (mean baseline),-0.0017,-0.0489,10.7233,13.6832,-0.0132,0.0166
1,Linear Regression,0.6135,0.5953,7.1146,8.4995,0.5957,0.0885


Saved comparison table to: E:\8thSem\student_performance_system\ml\results\task2_regression_results.csv


### Predicted vs. actual, and residuals

**Predicted-vs-actual**: every point is one test-set student; the dashed diagonal is where a perfect model's points would fall (predicted = actual). Points scattered fairly close around that line, in both directions, indicate a reasonably well-fitted model with no strong systematic bias.

**Residual plot**: for each test student, `residual = actual - predicted`, plotted against the predicted value. A well-behaved linear model's residuals should scatter randomly around the horizontal zero line with no visible pattern -- a curve or a funnel shape (residuals fanning wider at one end) would suggest the true relationship isn't fully linear, or that the model's errors get worse for certain kinds of students, which a single straight-line model can't fix on its own.

In [22]:
pred_vs_actual_df = pd.DataFrame({"actual": y2_test, "predicted": linear_reg_pred})

fig = px.scatter(
    pred_vs_actual_df, x="actual", y="predicted",
    title="Predicted vs. Actual Final Percentage (Linear Regression)",
)
axis_min = min(pred_vs_actual_df["actual"].min(), pred_vs_actual_df["predicted"].min())
axis_max = max(pred_vs_actual_df["actual"].max(), pred_vs_actual_df["predicted"].max())
fig.add_trace(go.Scatter(
    x=[axis_min, axis_max], y=[axis_min, axis_max], mode="lines",
    name="Perfect prediction", line=dict(dash="dash", color="gray"),
))
fig.update_layout(width=700, height=550)
fig.show()

In [23]:
residuals = y2_test - linear_reg_pred
residual_df = pd.DataFrame({"predicted": linear_reg_pred, "residual": residuals})

fig = px.scatter(
    residual_df, x="predicted", y="residual",
    title="Residual Plot (Linear Regression)",
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(width=700, height=500, yaxis_title="Actual - Predicted")
fig.show()

print(f"Residual mean: {residuals.mean():.4f} (should be close to 0 for an unbiased fit)")
print(f"Residual std:  {residuals.std():.4f}")

Residual mean: -0.6533 (should be close to 0 for an unbiased fit)
Residual std:  8.5219


### The regression equation

Written out in full, using the exact fitted coefficients, this model is:

```
final_percentage = intercept
                    + (coef_internal * internal_pct)
                    + (coef_attendance * attendance_pct)
                    + (coef_assignments * assignments_submitted)
                    + (coef_practical * practical_pct)
```

The cell below prints the real fitted numbers and a plain-language reading of each one.

In [24]:
equation_terms = " + ".join(
    f"({coef:+.4f} * {feature})" for feature, coef in zip(task2_features, linear_reg.coef_)
)
print(f"final_percentage = {linear_reg.intercept_:.4f} + {equation_terms}")
print()

coefficients_df2 = pd.DataFrame({
    "feature": task2_features,
    "coefficient": linear_reg.coef_.round(4),
}).sort_values("coefficient", key=abs, ascending=False)
display(coefficients_df2)

fig = px.bar(
    coefficients_df2, x="coefficient", y="feature", orientation="h",
    title="Linear Regression Coefficients -- Task 2 (deployed model)",
)
fig.update_layout(width=700, height=350)
fig.show()

print("Plain-language reading of each coefficient (holding other features constant):")
for feature, coef in zip(task2_features, linear_reg.coef_):
    print(f"  Each 1-unit increase in {feature:22s} is associated with a "
          f"{coef:+.4f} percentage-point change in predicted final marks.")
print(f"  Intercept ({linear_reg.intercept_:.4f}): the model's predicted final_percentage "
      "if every feature were 0 -- mostly a mathematical anchor point rather than a "
      "realistic scenario, since a real student would rarely have 0 on every input.")

final_percentage = 1.3800 + (+0.3367 * internal_pct) + (+0.1860 * attendance_pct) + (+0.9426 * assignments_submitted) + (+0.2740 * practical_pct)



,feature,coefficient
2,assignments_submitted,0.9426
0,internal_pct,0.3367
3,practical_pct,0.2740
1,attendance_pct,0.1860


Plain-language reading of each coefficient (holding other features constant):
  Each 1-unit increase in internal_pct           is associated with a +0.3367 percentage-point change in predicted final marks.
  Each 1-unit increase in attendance_pct         is associated with a +0.1860 percentage-point change in predicted final marks.
  Each 1-unit increase in assignments_submitted  is associated with a +0.9426 percentage-point change in predicted final marks.
  Each 1-unit increase in practical_pct          is associated with a +0.2740 percentage-point change in predicted final marks.
  Intercept (1.3800): the model's predicted final_percentage if every feature were 0 -- mostly a mathematical anchor point rather than a realistic scenario, since a real student would rarely have 0 on every input.


### Deployment justification (Task 2)

Linear Regression is the only real candidate for this task (the assignment specifies exactly one algorithm here), so "model selection" mainly means confirming it is actually worth deploying rather than assuming so:

- It explains roughly **61% of the variance** in final percentage (R² = 0.61, adjusted R² = 0.60 -- the two are close, meaning the model isn't overfit to the specific four features chosen), versus essentially 0% for the mean-only baseline.
- Its typical prediction is off by about **7 percentage points** (MAE), with RMSE around 8.5 -- meaningful, useful early-warning precision without pretending to be exact.
- 5-fold cross-validated R² (0.60, std 0.09) is close to the single test-set R² (0.61), showing the result is stable across different train/validation splits, not a lucky single split.
- Every coefficient has the intuitive sign (more marks, attendance, assignments, and practical work all *raise* predicted final marks; none are counter-intuitively negative), which is exactly the kind of sanity check a purely-metric-driven comparison can miss.
- As explained earlier, R² is capped well below 1.0 **by design** -- `final_percentage` is 60% driven by a hidden external-exam signal none of our features can see, so this ceiling reflects the honesty of the synthetic data, not a weak model.

In [25]:
config.ML_MODELS_DIR.mkdir(parents=True, exist_ok=True)

reg_model_path = config.ML_MODELS_DIR / "final_marks_regressor.pkl"
reg_metadata_path = config.ML_MODELS_DIR / "final_marks_regressor_metadata.json"

joblib.dump(linear_reg, reg_model_path)

reg_metadata = {
    "task": "final_marks_regression",
    "algorithm": type(linear_reg).__name__,
    "version": "1.0",
    "training_date": datetime.now(timezone.utc).isoformat(),
    "random_state": config.RANDOM_STATE,
    "features": task2_features,
    "target": task2_target,
    "requires_scaling": False,
    "hyperparameters": {k: v for k, v in linear_reg.get_params().items()},
    "equation": {
        "intercept": round(float(linear_reg.intercept_), 4),
        "coefficients": {f: round(float(c), 4) for f, c in zip(task2_features, linear_reg.coef_)},
    },
    "metrics": linear_reg_result,
    "all_candidates_compared": task2_results,
}

with open(reg_metadata_path, "w", encoding="utf-8") as f:
    json.dump(reg_metadata, f, indent=2, default=str)

print("Saved model to:", reg_model_path)
print("Saved metadata to:", reg_metadata_path)

Saved model to: E:\8thSem\student_performance_system\ml\models\final_marks_regressor.pkl
Saved metadata to: E:\8thSem\student_performance_system\ml\models\final_marks_regressor_metadata.json


## Task 3 — Student Segmentation (Unsupervised Clustering)

**The business problem**: group students into behaviourally meaningful segments based on how they perform, not just how much -- so an advisor can tell at a glance not only "who is struggling" but "who is struggling AND declining" versus "who is struggling BUT actively improving", which call for very different interventions.

**This task is fundamentally different from Tasks 1 and 2**: there is no `y`, no ground-truth label to predict, and therefore no accuracy/recall/R² to optimise against. K-Means instead DISCOVERS structure in the data by grouping similar students together -- we then interpret what it found afterwards, we do not tell it what to look for in advance.

In [26]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [27]:
task3_features = ["average_marks", "attendance_pct", "consistency", "improvement_rate"]
X3 = df[task3_features]

print("Feature summary:")
display(X3.describe().round(2))

Feature summary:


,average_marks,attendance_pct,consistency,improvement_rate
count,450.00,450.00,450.00,450.00
mean,50.47,69.39,8.33,-1.97
std,12.13,13.56,4.41,15.65
min,14.67,30.00,0.43,-53.72
25%,42.34,60.81,5.01,-13.19
50%,50.31,69.31,7.48,-1.60
75%,58.97,79.24,11.13,8.93
max,86.29,100.00,22.32,46.80


### Why K-Means requires scaling -- more essentially than any earlier model

K-Means assigns each point to its nearest centroid using Euclidean distance across ALL features at once. `average_marks`/`attendance_pct` range roughly 0-100, while `consistency` and `improvement_rate` naturally span a much narrower range. Without scaling, distance calculations would be dominated almost entirely by whichever feature happens to have the largest raw numbers -- not whichever feature is actually most meaningful for telling students apart. This is the same underlying reason Task 1's Logistic Regression needed scaling (both are distance/gradient based), but here it is not a secondary numerical-stability concern -- it directly determines which points get grouped together, so skipping it would silently produce meaningless clusters.

In [28]:
segmentation_scaler = StandardScaler()
X3_scaled = segmentation_scaler.fit_transform(X3)

print("Scaled features -- mean should be ~0, std should be ~1 per column:")
print(pd.DataFrame(X3_scaled, columns=task3_features).describe().round(2).loc[["mean", "std"]])

Scaled features -- mean should be ~0, std should be ~1 per column:
      average_marks  attendance_pct  consistency  improvement_rate
mean            0.0             0.0         -0.0               0.0
std             1.0             1.0          1.0               1.0


### Algorithm 5: K-Means Clustering

K-Means partitions data into `k` groups by iterating two steps until stable: (1) assign every point to whichever of `k` centroids it is closest to, and (2) move each centroid to the mean position of the points currently assigned to it. It is appropriate when you expect roughly round, similarly-sized groupings in the data and want a fast, simple, easily-interpreted partition (each cluster is just described by its centroid). Its main weaknesses: `k` must be chosen in advance (there is no single "correct" k the algorithm discovers on its own -- see the elbow method below), it assumes clusters are roughly spherical and similarly sized, which real data does not always respect, and its result depends on where the centroids start out randomly -- which is why `n_init` below reruns the whole algorithm from several random starting points and keeps the best result, rather than trusting a single run.

### Choosing k: the elbow method and silhouette score together

**Inertia** (used for the elbow method) is the sum of squared distances from every point to its assigned centroid -- it can only decrease (or stay flat) as `k` increases, since more clusters can always fit the data at least as well. The "elbow" is the `k` after which adding more clusters stops buying much of a reduction -- a sign that additional clusters are splitting real structure into finer pieces rather than finding genuinely new groups.

**Silhouette score** (from -1 to +1) measures, for each point, how much closer it is to its own cluster than to the nearest other cluster, averaged across all points -- higher means better-separated, more cohesive clusters. Unlike inertia, it does NOT automatically favour more clusters, so it is a useful second opinion alongside the elbow.

In [29]:
k_range = range(2, 9)
inertias = {}
silhouette_scores = {}

for k in k_range:
    kmeans_trial = KMeans(n_clusters=k, random_state=config.RANDOM_STATE, n_init=10)
    trial_labels = kmeans_trial.fit_predict(X3_scaled)
    inertias[k] = kmeans_trial.inertia_
    silhouette_scores[k] = silhouette_score(X3_scaled, trial_labels)

elbow_df = pd.DataFrame({
    "k": list(k_range),
    "inertia": [inertias[k] for k in k_range],
    "silhouette_score": [round(silhouette_scores[k], 4) for k in k_range],
})
display(elbow_df)

elbow_csv_path = RESULTS_DIR / "task3_clustering_k_selection.csv"
elbow_df.to_csv(elbow_csv_path, index=False)
print("Saved k-selection comparison table to:", elbow_csv_path)

fig = go.Figure()
fig.add_trace(go.Scatter(x=elbow_df["k"], y=elbow_df["inertia"], mode="lines+markers", name="Inertia"))
fig.update_layout(
    title="Elbow Method -- Inertia vs. k",
    xaxis_title="Number of clusters (k)", yaxis_title="Inertia (within-cluster sum of squares)",
    width=650, height=450,
)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=elbow_df["k"], y=elbow_df["silhouette_score"], mode="lines+markers",
                          name="Silhouette score", line=dict(color="darkorange")))
fig.update_layout(
    title="Silhouette Score vs. k",
    xaxis_title="Number of clusters (k)", yaxis_title="Silhouette score",
    width=650, height=450,
)
fig.show()

,k,inertia,silhouette_score
0,2,1314.702290,0.2372
1,3,1101.641174,0.2231
2,4,937.470798,0.2156
3,5,830.059534,0.2128
4,6,757.490975,0.2071
5,7,695.495345,0.2150
6,8,655.113527,0.1986


Saved k-selection comparison table to: E:\8thSem\student_performance_system\ml\results\task3_clustering_k_selection.csv


### Why k=4, even though the silhouette score is technically highest at k=2

The silhouette score peaks at k=2 (the coarsest possible split), then declines fairly steadily. Read on its own, that would suggest k=2. We do not use k=2, for a reason worth stating plainly rather than quietly picking whichever k "wins" on one number: a 2-way split can only ever separate "stronger" from "weaker" students -- it cannot distinguish a currently-weak-but-improving student from a weak-and-declining one, which is the entire practical point of this segmentation. Silhouette score alone does not know that some ways of grouping students are more USEFUL than others; it only measures geometric separation.

The elbow curve's rate of decrease slows down noticeably around k=4 (inertia drops roughly 15%+ per step for k=2 to k=4, then consistently under 12% afterwards) -- a defensible elbow. k=4's own silhouette score (reported below) is modest, not dramatic, and we say so honestly rather than overstating it: this dataset's students vary fairly continuously in ability (recall `ml/generate_data.py` generates every feature from ONE shared, continuous `latent_ability` variable, not from genuinely distinct subpopulations), so we would not expect sharply-separated natural clusters to exist in the first place. K-Means at k=4 still produces a genuinely useful, interpretable partition, as the cluster profiles below demonstrate -- it is just worth being upfront that "useful and interpretable" and "geometrically pristine" are not the same claim.

In [30]:
FINAL_K = 4

kmeans = KMeans(n_clusters=FINAL_K, random_state=config.RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X3_scaled)

final_silhouette = silhouette_score(X3_scaled, cluster_labels)
print(f"k = {FINAL_K}")
print(f"Silhouette score: {final_silhouette:.4f}")
print()
print("Cluster sizes:")
print(pd.Series(cluster_labels).value_counts().sort_index())

k = 4
Silhouette score: 0.2156

Cluster sizes:
0    125
1    115
2    115
3     95
Name: count, dtype: int64


### PCA: visualising 4 dimensions in 2

The clustering above happened in the full 4-dimensional SCALED feature space -- PCA is used only afterwards, purely to make a 2D scatter plot possible. It finds the two directions (linear combinations of the original four features) that capture the most variance in the data, and projects every point onto just those two. This is a visualisation aid, not part of the clustering itself -- the cluster ASSIGNMENTS come entirely from the 4D K-Means fit above; PCA never influences them.

We report exactly how much of the original structure survives the 2D projection, rather than implying the picture below is the whole story.

In [31]:
pca = PCA(n_components=2, random_state=config.RANDOM_STATE)
X3_pca = pca.fit_transform(X3_scaled)

print("Variance explained by each of the 2 components:", pca.explained_variance_ratio_.round(4))
print(f"Total variance captured in this 2D view: {pca.explained_variance_ratio_.sum():.1%} "
      f"(the remaining {1 - pca.explained_variance_ratio_.sum():.1%} of the original 4D structure "
      "is not visible in the plot below)")

pca_plot_df = pd.DataFrame(X3_pca, columns=["PC1", "PC2"])
pca_plot_df["cluster"] = cluster_labels.astype(str)

fig = px.scatter(
    pca_plot_df, x="PC1", y="PC2", color="cluster",
    title=f"Student Segments (k={FINAL_K}) -- PCA 2D Projection",
    category_orders={"cluster": sorted(pca_plot_df["cluster"].unique())},
)
fig.update_layout(width=750, height=550)
fig.show()

Variance explained by each of the 2 components: [0.4167 0.2547]
Total variance captured in this 2D view: 67.1% (the remaining 32.9% of the original 4D structure is not visible in the plot below)


### Naming the clusters

K-Means only ever outputs numeric cluster IDs (0, 1, 2, 3) -- those numbers carry no meaning of their own and are not guaranteed to appear in any particular order. Naming a cluster is a human interpretation step, done here by inspecting each cluster's CENTROID (its average `average_marks`, `attendance_pct`, `consistency`, `improvement_rate` in real, original units) and matching the most extreme, distinguishing trait to a descriptive label -- not by guessing, and not by hard-coding "cluster 0 = X" ahead of time (K-Means' cluster numbering can change between runs with different random seeds or data).

The naming rule below works by sequential elimination, most urgent signal first, so that no cluster can accidentally receive two labels: identify whichever remaining cluster has the clearest crisis signal (lowest average marks) first, then whichever remaining cluster has the clearest positive-trend signal (highest improvement rate), then whichever remaining cluster has the clearest instability signal (highest `consistency` value -- remember this column is a standard deviation, so a HIGH value means LESS consistent / more erratic performance across subjects, not more), and whatever is left over is the baseline strong-and-stable group.

In [32]:
centroids = X3.copy()
centroids["cluster"] = cluster_labels
centroid_stats = centroids.groupby("cluster").mean().round(2)
centroid_stats["student_count"] = centroids.groupby("cluster").size()
display(centroid_stats)

def name_clusters(stats: pd.DataFrame) -> dict:
    '''Assign a descriptive name to each cluster ID, by sequential
    elimination on the most distinguishing trait -- see the markdown
    above for the full reasoning.'''
    remaining = list(stats.index)
    names = {}

    needs_intervention_id = stats.loc[remaining, "average_marks"].idxmin()
    names[needs_intervention_id] = "Needs Intervention"
    remaining.remove(needs_intervention_id)

    improving_id = stats.loc[remaining, "improvement_rate"].idxmax()
    names[improving_id] = "Improving Students"
    remaining.remove(improving_id)

    inconsistent_id = stats.loc[remaining, "consistency"].idxmax()
    names[inconsistent_id] = "Inconsistent Performers"
    remaining.remove(inconsistent_id)

    for leftover_id in remaining:
        names[leftover_id] = "Consistent High Performers"

    return names

cluster_names = name_clusters(centroid_stats)
print("Cluster ID -> Name mapping:")
for cluster_id, name in sorted(cluster_names.items()):
    print(f"  Cluster {cluster_id}: {name}")

centroid_stats["cluster_name"] = centroid_stats.index.map(cluster_names)
display(centroid_stats)

,average_marks,attendance_pct,consistency,improvement_rate,student_count
cluster,,,,,
0,62.08,82.96,5.88,-3.60,125
1,38.29,58.48,8.62,-14.74,115
2,47.80,62.47,6.05,11.00,115
3,53.17,73.12,13.97,-0.06,95


Cluster ID -> Name mapping:
  Cluster 0: Consistent High Performers
  Cluster 1: Needs Intervention
  Cluster 2: Improving Students
  Cluster 3: Inconsistent Performers


,average_marks,attendance_pct,consistency,improvement_rate,student_count,cluster_name
cluster,,,,,,
0,62.08,82.96,5.88,-3.60,125,Consistent High Performers
1,38.29,58.48,8.62,-14.74,115,Needs Intervention
2,47.80,62.47,6.05,11.00,115,Improving Students
3,53.17,73.12,13.97,-0.06,95,Inconsistent Performers


In [33]:
pca_plot_df["cluster_name"] = pca_plot_df["cluster"].astype(int).map(cluster_names)

fig = px.scatter(
    pca_plot_df, x="PC1", y="PC2", color="cluster_name",
    title=f"Student Segments (k={FINAL_K}) -- Named, PCA 2D Projection",
)
fig.update_layout(width=800, height=550)
fig.show()

### Cluster profiles

*(the real numbers behind this summary are printed in the table two cells above; here is the plain-language reading of each one)*

- **Consistent High Performers** -- the highest average marks and highest attendance of any group, and the LOWEST `consistency` value (meaning the most stable, least erratic performance across different subjects). A mildly negative improvement rate is present but small next to the other groups' swings -- this group is doing well and staying steady, not necessarily still climbing.
- **Needs Intervention** -- the lowest average marks, lowest attendance, AND the sharpest negative improvement rate of any group. Every signal points the same direction at once, which is what makes this group the clearest priority for support.
- **Improving Students** -- moderate current marks and attendance, but by far the strongest POSITIVE improvement rate of any group, and reasonably consistent. This is the group a purely current-snapshot view (ignoring trend) would risk mis-classifying as simply "average" -- segmentation that includes `improvement_rate` is what surfaces them separately from a merely-average, non-improving student.
- **Inconsistent Performers** -- decent average marks and attendance, but by far the highest `consistency` value (i.e. the LEAST consistent group -- their performance swings noticeably between subjects), with a roughly flat improvement trend. This group's risk is unpredictability, not low ability.

### Saving the deployed model

K-Means and PCA have no "predict a known outcome" metric to report the way Tasks 1 and 2 did -- there is no ground truth to compare against. What gets saved here is the fitted clustering itself (so `modules/ml_predictions.py` can assign a NEW student to one of these four named segments later, without retraining), its scaler, and a metadata file recording the k selection reasoning, the final silhouette score, and the cluster-name mapping discovered above.

In [34]:
config.ML_MODELS_DIR.mkdir(parents=True, exist_ok=True)

cluster_model_path = config.ML_MODELS_DIR / "student_segmentation.pkl"
cluster_scaler_path = config.ML_MODELS_DIR / "segmentation_scaler.pkl"
cluster_metadata_path = config.ML_MODELS_DIR / "student_segmentation_metadata.json"

joblib.dump(kmeans, cluster_model_path)
joblib.dump(segmentation_scaler, cluster_scaler_path)

cluster_metadata = {
    "task": "student_segmentation",
    "algorithm": type(kmeans).__name__,
    "version": "1.0",
    "training_date": datetime.now(timezone.utc).isoformat(),
    "random_state": config.RANDOM_STATE,
    "features": task3_features,
    "requires_scaling": True,
    "k": FINAL_K,
    "k_selection": {
        "silhouette_by_k": {str(k): round(v, 4) for k, v in silhouette_scores.items()},
        "inertia_by_k": {str(k): round(v, 2) for k, v in inertias.items()},
        "chosen_k_silhouette": round(final_silhouette, 4),
        "note": "silhouette peaks at k=2, but k=4 was chosen for practical segment usefulness -- see notebook markdown.",
    },
    "pca_explained_variance_ratio_2d": pca.explained_variance_ratio_.round(4).tolist(),
    "cluster_names": {str(k): v for k, v in cluster_names.items()},
    "cluster_centroids": json.loads(centroid_stats.drop(columns="cluster_name").to_json(orient="index")),
}

with open(cluster_metadata_path, "w", encoding="utf-8") as f:
    json.dump(cluster_metadata, f, indent=2, default=str)

print("Saved model to:", cluster_model_path)
print("Saved scaler to:", cluster_scaler_path)
print("Saved metadata to:", cluster_metadata_path)

Saved model to: E:\8thSem\student_performance_system\ml\models\student_segmentation.pkl
Saved scaler to: E:\8thSem\student_performance_system\ml\models\segmentation_scaler.pkl
Saved metadata to: E:\8thSem\student_performance_system\ml\models\student_segmentation_metadata.json
